In [ ]:
import polars as pl
from pathlib import Path
import shutil

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
#### CHANGE THIS NAME ####
brms_run_name = "brms_polarity_sensitivity"
UNFILTERED = False  # True: all genomes | False: seed 1 (main data)
##########################


PRIMARY_SEED = 1  # chosen a priori, before any results
DRAW_DIR = BASE / "brms_draws"  # only used to verify the seed rebuild

print(
    f"Run: {brms_run_name} ({'all genomes' if UNFILTERED else f'seed {PRIMARY_SEED}'})"
)

groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]
polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opp",
    "-+": "opp",
}
polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "conv",
    "-+": "div",
}
allow_polarity = set(polarity_3bin.keys())

In [ ]:
trna_labels = {
    "TA",
    "TASX",
    "ASX",
    "TC",
    "TD",
    "TE",
    "TER",
    "TF",
    "TFM",
    "TG",
    "TGLX",
    "TH",
    "TI",
    "TK",
    "TL1|2",
    "TM",
    "TN",
    "TP",
    "TQ",
    "TR",
    "TRMR",
    "TRNX",
    "TS1|2",
    "TSEC",
    "TSUP",
    "TT",
    "TU",
    "TV",
    "TW",
    "TX",
    "TXLE",
    "XLE",
    "TY",
}

rrna_labels = {
    "RNR12",
    "RNR15",
    "RNR16",
    "RNR18",
    "RNR21",
    "RNR23",
    "RNR26",
    "RNR28",
    "RNR4",
    "RNR4.5",
    "RNR5",
    "RNR9",
    "RNRL",
    "RNRS",
    "RRNA",
    "RRN3",
    "RRN7",
    "RRNF",
    "RRF",
}

intronic_labels = {
    "GIY-YIG",
    "I-END",
    "I-MAT",
    "I-RT",
    "LAGLIDADG",
    "MAT",
    "NAT",
    "RT",
    "LAG-END",
    "MATK",
    "MATR",
    "END-LAG",
}

In [ ]:
# Output columns for <brms_run_name>/brms_input.tsv (same as the draws)
brms_cols = [
    "AN",
    "ID",
    "polarity_3bin",
    "igr_len",
    "flanking_types",
    "ncbi_taxid",
    "log10_igr_len",
    "genome_length",
    "log10_genome_length",
    "log10_geom_mean_flanking_gene_len",
    "log10_min_flanking_gene_len",
    "log10_max_flanking_gene_len",
    "sup_len",
    "sdw_len",
    "ncbi_name",
]

In [ ]:
all_data = {}
for g in groups:
    gdir = BASE / g
    igr = pl.read_csv(gdir / f"{g}_summary_igr.tsv", separator="\t")
    igr = igr.filter(pl.col("Length") > 0)  # Filter out zero-length IGRs
    tsv = (
        pl.read_csv(gdir / f"{g}.tsv", separator="\t")
        .select(["AN", "ncbi_taxid", "Genome_length", "ncbi_name"])
        .rename({"Genome_length": "genome_length"})
    )

    igr = igr.join(tsv, on="AN", how="inner")

    for side, col in (("sup", "source_up"), ("sdw", "source_dw")):
        igr = (
            igr.with_columns(
                pl.col(col)
                .str.split_exact("|", 5)
                .struct.rename_fields(
                    [
                        f"{side}_an",
                        f"{side}_type",
                        f"{side}_start",
                        f"{side}_end",
                        f"{side}_polarity",
                        f"{side}_gff_id",
                    ]
                )
                .alias("parts")
            )
            .unnest("parts")
            .with_columns(
                pl.col(f"{side}_start").cast(pl.Int64),
                pl.col(f"{side}_end").cast(pl.Int64),
            )
            .drop([f"{side}_an", f"{side}_type", f"{side}_gff_id"])
        )

    before = igr.height
    igr = igr.filter(pl.col("Polarity").is_in(allow_polarity))
    print(
        f"{g} - {before - igr.height:,} IGRs dropped due to polarity class not in {allow_polarity})"
    )

    igr = igr.with_columns(
        pl.col("Polarity").replace_strict(polarity_3bin).alias("polarity_3bin"),
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin"),
    )
    igr = igr.with_columns(
        (pl.col("sup_end") - pl.col("sup_start") + 1).alias("sup_len"),
        (pl.col("sdw_end") - pl.col("sdw_start") + 1).alias("sdw_len"),
    )
    igr = igr.with_columns(
        pl.col("sup_len").fill_null(pl.col("sdw_len")).alias("sup_len"),
        pl.col("sdw_len").fill_null(pl.col("sup_len")).alias("sdw_len"),
    )
    igr = igr.with_columns(
        ((pl.col("sup_len") + pl.col("sdw_len")) / 2).alias("mean_flanking_gene_len"),
        ((pl.col("sup_len").log10() + pl.col("sdw_len").log10()) / 2).alias(
            "log10_geom_mean_flanking_gene_len"
        ),
        pl.min_horizontal("sup_len", "sdw_len")
        .log10()
        .alias("log10_min_flanking_gene_len"),
        pl.max_horizontal("sup_len", "sdw_len")
        .log10()
        .alias("log10_max_flanking_gene_len"),
    )
    igr = igr.with_columns(
        pl.col("Length").log10().alias("log10_igr_len"),
        pl.col("genome_length").log10().alias("log10_genome_length"),
    )
    igr = igr.rename({"Length": "igr_len"})

    igr = igr.with_columns(
        pl.col("UP").str.replace_all(r"^MIT-|^PLT-", "").alias("label_up"),
        pl.col("DOWN").str.replace_all(r"^MIT-|^PLT-", "").alias("label_dw"),
    )
    igr = igr.with_columns(
        pl.when(pl.col("label_up").is_in(trna_labels))
        .then(pl.lit("trna"))
        .when(pl.col("label_up").is_in(rrna_labels))
        .then(pl.lit("rrna"))
        .otherwise(pl.lit("gene"))
        .alias("type_up"),
        pl.when(pl.col("label_dw").is_in(trna_labels))
        .then(pl.lit("trna"))
        .when(pl.col("label_dw").is_in(rrna_labels))
        .then(pl.lit("rrna"))
        .otherwise(pl.lit("gene"))
        .alias("type_dw"),
    )
    igr = igr.with_columns(
        pl.when(pl.col("type_up") <= pl.col("type_dw"))
        .then(pl.col("type_up") + "-" + pl.col("type_dw"))
        .otherwise(pl.col("type_dw") + "-" + pl.col("type_up"))
        .alias("flanking_types")
    )
    print(f"{g} {igr['AN'].n_unique():>7,} genomes  {igr.height:>10,} IGRs")

    all_data[g] = igr

## Genome selection

In [ ]:
def draw_genus_representatives(g, seed):
    # Must stay identical to the draws/refits notebooks
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t").select(
        ["AN", "ncbi_genus"]
    )
    return (
        tsv.sample(fraction=1.0, shuffle=True, seed=seed)
        .group_by("ncbi_genus", maintain_order=True)
        .head(1)
    )


def check_empty_genus(g):
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t").select(
        ["AN", "ncbi_genus"]
    )
    empty = tsv.filter(
        pl.col("ncbi_genus").is_null() | (pl.col("ncbi_genus").str.strip_chars() == "")
    )
    if empty.height:
        print(
            f"  WARNING [{g}] {empty.height:,} genome(s) with empty ncbi_genus "
            f"(collapsed into one 'genus' by the draw), e.g. {empty['AN'].head(5).to_list()}"
        )

In [ ]:
selected = {}
representatives = {}

for g in groups:
    data = all_data[g]

    if UNFILTERED:
        selected[g] = data
        print(
            f"{g:16s} {data['AN'].n_unique():>6,} genomes  {data.height:>9,} IGRs  (all genomes)"
        )
        continue

    check_empty_genus(g)
    pick = draw_genus_representatives(g, PRIMARY_SEED)
    ans = pick["AN"].to_list()
    representatives[g] = pick
    selected[g] = data.filter(pl.col("AN").is_in(ans))

    # Representatives without any IGR left after the length/polarity filters
    no_igr = pick.filter(~pl.col("AN").is_in(selected[g]["AN"].unique().to_list()))
    if no_igr.height:
        print(
            f"  NOTE [{g}] {no_igr.height:,} of {pick.height:,} genera drew a genome "
            f"with no IGRs (genus absent from the input), e.g. "
            f"{no_igr['ncbi_genus'].head(5).to_list()}"
        )

    # Verify against the brms_draws seed reference
    ref_dir = DRAW_DIR / g / f"seed_{PRIMARY_SEED}"
    ref_reps = ref_dir / "genus_representatives.tsv"
    ref_input = ref_dir / "brms_input.tsv"
    status = "WARNING: no reference found, not verified"
    if ref_reps.exists():
        ref_ans = set(pl.read_csv(ref_reps, separator="\t")["AN"])
        if ref_ans != set(ans):
            raise ValueError(
                f"{g}: rebuilt seed {PRIMARY_SEED} representatives differ from "
                f"{ref_reps} ({len(ref_ans ^ set(ans))} genomes differ)"
            )
        status = "representatives match"
        if ref_input.exists():
            ref_ids = pl.read_csv(ref_input, separator="\t", columns=["AN", "ID"])
            new_ids = selected[g].select(["AN", "ID"])
            if (
                ref_ids.height != new_ids.height
                or ref_ids.join(new_ids, on=["AN", "ID"], how="anti").height
            ):
                raise ValueError(f"{g}: rebuilt IGRs differ from {ref_input}")
            status += ", IGRs match"
    print(f"{g:16s} {len(ans):>6,} genomes  {selected[g].height:>9,} IGRs  ({status})")

## Write brms input

In [ ]:
for g in groups:
    gdir = BASE / g
    brms_folder = gdir / brms_run_name
    brms_folder.mkdir(exist_ok=True)

    # Whole tree; the R script prunes it with keep.tip(tree, levels(igr$ncbi_taxid))
    shutil.copy(gdir / "tree.nwk", brms_folder / "tree.nwk")

    data = selected[g].select(brms_cols)

    # round numerical columns to 4 decimal places
    data = data.with_columns(pl.col(pl.Float32, pl.Float64).round(4))
    data.write_csv(brms_folder / "brms_input.tsv", separator="\t")

    if not UNFILTERED:
        representatives[g].write_csv(
            brms_folder / "genus_representatives.tsv", separator="\t"
        )

    print(
        f"Prepared {g:16s} -> {brms_folder.relative_to(BASE)}  ({data.height:,} IGRs)"
    )